In [ ]:
import os
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import re
import openpyxl
import numpy as np

# #%pip install pygwalker
# import pygwalker as pyg

ModuleNotFoundError: No module named 'pygwalker'

## Data Importing

In [ ]:
df_sales=pd.read_excel(r"C:\Users\vikram.vadhirajan\OneDrive - Trico\Documents - Product Support India Romania\Catalog\Score card\EPR\20250416\OR_Sales.xlsx",sheet_name='Sales')
df_sales.head(5)

,BusinessUnit,CustomerDesc,CustomerState,ProductCode,ProductDesc,QTY SHIPPED,Product Category
0,BPI,LES SCHWAB TIRE CENTERS,OR,RC12718C,PREMIUM COATED LOADED CALIPER,534.0,Calipers
1,BPI,LES SCHWAB TIRE CENTERS,OR,RC11711C,PREMIUM COATED LOADED CALIPER,3275.0,Calipers
2,BPI,LES SCHWAB TIRE CENTERS,OR,RC12041C,PREMIUM COATED LOADED CALIPER,1654.0,Calipers
3,BPI,NAPA PORTLAND DC,OR,62-48880687P,BRAKE ROTOR,52.0,Rotors
4,BPI,LES SCHWAB TIRE CENTERS,OR,RC12615CS,PREMIUM COATED LOADED CALIPER,7438.0,Calipers


In [3]:
ProductMap=r"C:\Users\vikram.vadhirajan\OneDrive - Trico\Documents - Product Support India Romania\Catalog\Score card\EPR\20250416\Product_Map.xlsx"

In [4]:
df_PM=pd.read_excel(ProductMap,sheet_name='Product Map')
df_PM.head(5)

,Customer,Product Description,Product Name
0,BPI,PREMIUM COATED LOADED CALIPER,Calipers
1,BPI,PREMIUM COATED LOADED CALIPER,Calipers
2,BPI,PREMIUM COATED LOADED CALIPER,Calipers
3,BPI,BRAKE ROTOR,Rotors
4,BPI,PREMIUM COATED LOADED CALIPER,Calipers


In [5]:
df_Exclusions=pd.read_excel(ProductMap,sheet_name='Exclusion',usecols="B:D",skiprows=1)
df_Exclusions

,BU,Customers,BUCUSTOMER
0,Champ,AUTOZONE #5974,ChampAUTOZONE #5974
1,Champ,NAPA PORTLAND PDC,ChampNAPA PORTLAND PDC
2,Carter,ADVANCE AUTO PARTS WHSE,CarterADVANCE AUTO PARTS WHSE
3,Carter,AUTOZONE INC,CarterAUTOZONE INC
4,Carter,CARQUEST POR 0036,CarterCARQUEST POR 0036
5,BPI,ADVANCE - CARQUEST POR 0036,BPIADVANCE - CARQUEST POR 0036
6,BPI,NAPA PORTLAND DC,BPINAPA PORTLAND DC
7,IBI,ADVANCE - CARQUEST POR 0036,IBIADVANCE - CARQUEST POR 0036
8,IBI,ADVANCE AUTO PARTS,IBIADVANCE AUTO PARTS
9,TRICO,ADVANCE AUTO PARTS,TRICOADVANCE AUTO PARTS


In [6]:
df_Packinginfo=pd.read_excel(ProductMap,sheet_name="Type")
df_Packinginfo.head(5)

,BU Legacy,Product,Concat,Line Number,Material Category,Qty,Average weight (lb),Unnamed: 7
0,ASC,ASC,ASCASC,line number:1,Corrugated Cardboard,1.0,0.270,ASC
1,ASC,ASC,ASCASC,line number:2,Corrugated Cardboard,1.0,0.135,ASC
2,ASC,ASC,ASCASC,line number:3,Kraft Paper,1.0,0.590,ASC
3,ASC,ASC,ASCASC,line number:4,Other Paper Laminates,1.0,0.001,ASC
4,ASC,ASC,ASCASC,line number:5,Paperboard,1.0,0.006,ASC


In [7]:
# Customer Segmentation
SpecificList=['Hopkins', 'BPI', 'Centric','Carter','Horizon','Cardone']
GenericList=["TRICO","Champ","Autolite","FRAM","IBI",'AVM']

In [8]:
for i in range(len(df_sales)):
    business_unit = df_sales.BusinessUnit[i]
    product_desc = df_sales.ProductDesc[i]

    if business_unit in SpecificList:
        try:
            product_name = df_PM.loc[
                df_PM['Product Description'] == product_desc, 'Product Name'
            ].values[0]
            df_sales.loc[i, 'ProductName'] = product_name
        except Exception:
            df_sales.loc[i, 'ProductName'] = "ProductNot Found in Mapping"

    elif business_unit in GenericList:
        if business_unit == 'FRAM' and 'AIR' in product_desc:
            df_sales.loc[i, 'ProductName'] = ""
        elif business_unit == 'Champ' and 'AIR' in product_desc:
            df_sales.loc[i, 'ProductName'] = "Air Filter"
        elif business_unit == 'Champ' and 'OIL' in product_desc:
            df_sales.loc[i, 'ProductName'] = "Oil Filter"
        else:
            try:
                product_name = df_PM.loc[
                    df_PM['Customer'] == business_unit, 'Product Name'
                ].values[0]
                df_sales.loc[i, 'ProductName'] = product_name
            except Exception:
                df_sales.loc[i, 'ProductName'] = "ProductNot Found in Mapping"
    else:
        df_sales.loc[i, 'ProductName'] = ""


In [9]:
df_sales

,BusinessUnit,CustomerDesc,CustomerState,ProductCode,ProductDesc,QTY SHIPPED,Product Category,ProductName
0,BPI,LES SCHWAB TIRE CENTERS,OR,RC12718C,PREMIUM COATED LOADED CALIPER,534.0,Calipers,Calipers
1,BPI,LES SCHWAB TIRE CENTERS,OR,RC11711C,PREMIUM COATED LOADED CALIPER,3275.0,Calipers,Calipers
2,BPI,LES SCHWAB TIRE CENTERS,OR,RC12041C,PREMIUM COATED LOADED CALIPER,1654.0,Calipers,Calipers
3,BPI,NAPA PORTLAND DC,OR,62-48880687P,BRAKE ROTOR,52.0,Rotors,Rotors
4,BPI,LES SCHWAB TIRE CENTERS,OR,RC12615CS,PREMIUM COATED LOADED CALIPER,7438.0,Calipers,Calipers
...,...,...,...,...,...,...,...,...
32920,TRICO,NAPA PORTLAND,OR,35-210E,"21"" TRICO ICE BEAM BLADE",145.0,NaN,Trico
32921,TRICO,NAPA PORTLAND,OR,35-260E,"26"" TRICO ICE BEAM BLADE",195.0,NaN,Trico
32922,TRICO,NAPA PORTLAND,OR,35-280E,"28"" TRICO ICE BEAM BLADE",70.0,NaN,Trico
32923,TRICO,NAPA PORTLAND,OR,35-200E,"20"" TRICO ICE BEAM BLADE",360.0,NaN,Trico


In [10]:
df_sales=df_sales[['CustomerDesc','ProductCode', 'BusinessUnit','ProductDesc','CustomerState', 'ProductName', 'QTY SHIPPED']] #Removing unwanted columns from the original dataset
df_sales['Key']=df_sales['BusinessUnit']+df_sales['ProductName'] #Creating Key column for merging with Product Map

In [11]:
df_final=df_sales.merge(df_Packinginfo, how='left', left_on='Key', right_on='Concat', indicator=True) # merging with Product Map to get the packing info
df_final['BUCUSTOMER']=df_final['BusinessUnit']+df_final['CustomerDesc']

In [12]:
df_final

,CustomerDesc,ProductCode,BusinessUnit,ProductDesc,CustomerState,ProductName,QTY SHIPPED,Key,BU Legacy,Product,Concat,Line Number,Material Category,Qty,Average weight (lb),Unnamed: 7,_merge,BUCUSTOMER
0,LES SCHWAB TIRE CENTERS,RC12718C,BPI,PREMIUM COATED LOADED CALIPER,OR,Calipers,534.0,BPICalipers,BPI,Calipers,BPICalipers,line number:1,Corrugated Cardboard,1.0,0.8000,Calipers,both,BPILES SCHWAB TIRE CENTERS
1,LES SCHWAB TIRE CENTERS,RC12718C,BPI,PREMIUM COATED LOADED CALIPER,OR,Calipers,534.0,BPICalipers,BPI,Calipers,BPICalipers,line number:2,Other Paper Laminates,1.0,0.0050,Calipers,both,BPILES SCHWAB TIRE CENTERS
2,LES SCHWAB TIRE CENTERS,RC12718C,BPI,PREMIUM COATED LOADED CALIPER,OR,Calipers,534.0,BPICalipers,BPI,Calipers,BPICalipers,line number:3,HDPE (#2)/LDPE (#4) Flexible and Film Items,1.0,0.0529,Calipers,both,BPILES SCHWAB TIRE CENTERS
3,LES SCHWAB TIRE CENTERS,RC11711C,BPI,PREMIUM COATED LOADED CALIPER,OR,Calipers,3275.0,BPICalipers,BPI,Calipers,BPICalipers,line number:1,Corrugated Cardboard,1.0,0.8000,Calipers,both,BPILES SCHWAB TIRE CENTERS
4,LES SCHWAB TIRE CENTERS,RC11711C,BPI,PREMIUM COATED LOADED CALIPER,OR,Calipers,3275.0,BPICalipers,BPI,Calipers,BPICalipers,line number:2,Other Paper Laminates,1.0,0.0050,Calipers,both,BPILES SCHWAB TIRE CENTERS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
480852,NAPA PORTLAND,35-180E,TRICO,"18"" TRICO ICE BEAM BLADE",OR,Trico,190.0,TRICOTrico,TRICO,Trico,TRICOTrico,line number:243,Paperboard,1.0,0.1050,Trico,both,TRICONAPA PORTLAND
480853,NAPA PORTLAND,35-180E,TRICO,"18"" TRICO ICE BEAM BLADE",OR,Trico,190.0,TRICOTrico,TRICO,Trico,TRICOTrico,line number:244,Paperboard,1.0,0.1050,Trico,both,TRICONAPA PORTLAND
480854,NAPA PORTLAND,35-180E,TRICO,"18"" TRICO ICE BEAM BLADE",OR,Trico,190.0,TRICOTrico,TRICO,Trico,TRICOTrico,line number:245,Paperboard,1.0,0.1050,Trico,both,TRICONAPA PORTLAND
480855,NAPA PORTLAND,35-180E,TRICO,"18"" TRICO ICE BEAM BLADE",OR,Trico,190.0,TRICOTrico,TRICO,Trico,TRICOTrico,line number:246,Paperboard,1.0,0.1050,Trico,both,TRICONAPA PORTLAND


In [13]:
df_final=df_final.merge(df_Exclusions, how='left', indicator='Source') #merginging with Exclusions to get the exclusions

In [14]:
df_final['CustomerType'] = np.where(df_final['Source']=='both', 'Exclusion', 'Inclusion')
df_final['TotalWeight']=df_final['QTY SHIPPED']*df_final['Qty']*df_final['Average weight (lb)']

In [15]:
df_final

,CustomerDesc,ProductCode,BusinessUnit,ProductDesc,CustomerState,ProductName,QTY SHIPPED,Key,BU Legacy,Product,...,Qty,Average weight (lb),Unnamed: 7,_merge,BUCUSTOMER,BU,Customers,Source,CustomerType,TotalWeight
0,LES SCHWAB TIRE CENTERS,RC12718C,BPI,PREMIUM COATED LOADED CALIPER,OR,Calipers,534.0,BPICalipers,BPI,Calipers,...,1.0,0.8000,Calipers,both,BPILES SCHWAB TIRE CENTERS,NaN,NaN,left_only,Inclusion,427.2000
1,LES SCHWAB TIRE CENTERS,RC12718C,BPI,PREMIUM COATED LOADED CALIPER,OR,Calipers,534.0,BPICalipers,BPI,Calipers,...,1.0,0.0050,Calipers,both,BPILES SCHWAB TIRE CENTERS,NaN,NaN,left_only,Inclusion,2.6700
2,LES SCHWAB TIRE CENTERS,RC12718C,BPI,PREMIUM COATED LOADED CALIPER,OR,Calipers,534.0,BPICalipers,BPI,Calipers,...,1.0,0.0529,Calipers,both,BPILES SCHWAB TIRE CENTERS,NaN,NaN,left_only,Inclusion,28.2486
3,LES SCHWAB TIRE CENTERS,RC11711C,BPI,PREMIUM COATED LOADED CALIPER,OR,Calipers,3275.0,BPICalipers,BPI,Calipers,...,1.0,0.8000,Calipers,both,BPILES SCHWAB TIRE CENTERS,NaN,NaN,left_only,Inclusion,2620.0000
4,LES SCHWAB TIRE CENTERS,RC11711C,BPI,PREMIUM COATED LOADED CALIPER,OR,Calipers,3275.0,BPICalipers,BPI,Calipers,...,1.0,0.0050,Calipers,both,BPILES SCHWAB TIRE CENTERS,NaN,NaN,left_only,Inclusion,16.3750
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
480852,NAPA PORTLAND,35-180E,TRICO,"18"" TRICO ICE BEAM BLADE",OR,Trico,190.0,TRICOTrico,TRICO,Trico,...,1.0,0.1050,Trico,both,TRICONAPA PORTLAND,TRICO,NAPA PORTLAND,both,Exclusion,19.9500
480853,NAPA PORTLAND,35-180E,TRICO,"18"" TRICO ICE BEAM BLADE",OR,Trico,190.0,TRICOTrico,TRICO,Trico,...,1.0,0.1050,Trico,both,TRICONAPA PORTLAND,TRICO,NAPA PORTLAND,both,Exclusion,19.9500
480854,NAPA PORTLAND,35-180E,TRICO,"18"" TRICO ICE BEAM BLADE",OR,Trico,190.0,TRICOTrico,TRICO,Trico,...,1.0,0.1050,Trico,both,TRICONAPA PORTLAND,TRICO,NAPA PORTLAND,both,Exclusion,19.9500
480855,NAPA PORTLAND,35-180E,TRICO,"18"" TRICO ICE BEAM BLADE",OR,Trico,190.0,TRICOTrico,TRICO,Trico,...,1.0,0.1050,Trico,both,TRICONAPA PORTLAND,TRICO,NAPA PORTLAND,both,Exclusion,19.9500


In [16]:
df_final=df_final[['CustomerDesc','CustomerType','BusinessUnit','ProductCode', 'ProductDesc', 'ProductName', 'Line Number','CustomerState',
       'QTY SHIPPED', 'Material Category', 'Average weight (lb)',  'Qty','TotalWeight']]

In [17]:
df_final

,CustomerDesc,CustomerType,BusinessUnit,ProductCode,ProductDesc,ProductName,Line Number,CustomerState,QTY SHIPPED,Material Category,Average weight (lb),Qty,TotalWeight
0,LES SCHWAB TIRE CENTERS,Inclusion,BPI,RC12718C,PREMIUM COATED LOADED CALIPER,Calipers,line number:1,OR,534.0,Corrugated Cardboard,0.8000,1.0,427.2000
1,LES SCHWAB TIRE CENTERS,Inclusion,BPI,RC12718C,PREMIUM COATED LOADED CALIPER,Calipers,line number:2,OR,534.0,Other Paper Laminates,0.0050,1.0,2.6700
2,LES SCHWAB TIRE CENTERS,Inclusion,BPI,RC12718C,PREMIUM COATED LOADED CALIPER,Calipers,line number:3,OR,534.0,HDPE (#2)/LDPE (#4) Flexible and Film Items,0.0529,1.0,28.2486
3,LES SCHWAB TIRE CENTERS,Inclusion,BPI,RC11711C,PREMIUM COATED LOADED CALIPER,Calipers,line number:1,OR,3275.0,Corrugated Cardboard,0.8000,1.0,2620.0000
4,LES SCHWAB TIRE CENTERS,Inclusion,BPI,RC11711C,PREMIUM COATED LOADED CALIPER,Calipers,line number:2,OR,3275.0,Other Paper Laminates,0.0050,1.0,16.3750
...,...,...,...,...,...,...,...,...,...,...,...,...,...
480852,NAPA PORTLAND,Exclusion,TRICO,35-180E,"18"" TRICO ICE BEAM BLADE",Trico,line number:243,OR,190.0,Paperboard,0.1050,1.0,19.9500
480853,NAPA PORTLAND,Exclusion,TRICO,35-180E,"18"" TRICO ICE BEAM BLADE",Trico,line number:244,OR,190.0,Paperboard,0.1050,1.0,19.9500
480854,NAPA PORTLAND,Exclusion,TRICO,35-180E,"18"" TRICO ICE BEAM BLADE",Trico,line number:245,OR,190.0,Paperboard,0.1050,1.0,19.9500
480855,NAPA PORTLAND,Exclusion,TRICO,35-180E,"18"" TRICO ICE BEAM BLADE",Trico,line number:246,OR,190.0,Paperboard,0.1050,1.0,19.9500


In [18]:
df_mapped = df_final.pivot_table(
    index=['CustomerType','Material Category'],
    columns=['BusinessUnit'],
    values='TotalWeight',
    aggfunc='sum'
).reset_index()

df_mapped.iloc[:, 1:] = df_mapped.iloc[:, 1:].round(1)

df_mapped


BusinessUnit,CustomerType,Material Category,AVM,Autolite,BPI,Cardone,Carter,Centric,Champ,FRAM,Hopkins,Horizon,IBI,TRICO
0,Exclusion,Corrugated Cardboard,NaN,NaN,24042.0,17786.5,NaN,NaN,29.2,NaN,NaN,NaN,1.7,NaN
1,Exclusion,Corrugated Laminates,4505.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3536891.1
2,Exclusion,HDPE (#2)/LDPE (#4) Flexible and Film Items,1095.2,NaN,877.1,150.5,NaN,NaN,NaN,NaN,NaN,NaN,0.1,365092.0
3,Exclusion,Kraft Paper,NaN,NaN,NaN,275.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Exclusion,Other Paper Laminates,33.2,NaN,470.5,NaN,NaN,NaN,0.2,NaN,NaN,NaN,0.0,24326.1
5,Exclusion,PET (#1) - Other Rigid Items,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,156039.3
6,Exclusion,PS (#6) Colored Expanded/Foamed Cushioning,NaN,NaN,NaN,65.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Exclusion,Paper - Small Format,NaN,NaN,NaN,108.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Exclusion,Paper for General Use,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,75018.9
9,Exclusion,Paperboard,NaN,NaN,1030.1,NaN,NaN,NaN,18.5,NaN,NaN,NaN,0.4,4845845.8


In [24]:
pyg.walk(df_final)


NameError: name 'pyg' is not defined

In [20]:
FileName=r"C:\Users\vikram.vadhirajan\OneDrive - Trico\Documents - Product Support India Romania\Catalog\Score card\EPR\20250416\PackingInfo_Consolidated.xlsx"
with pd.ExcelWriter(FileName) as writer:  # doctest: +SKIP
    df_final.to_excel(writer,index=False, sheet_name='Raw')
    df_mapped.to_excel(writer,index=False, sheet_name='Summary')

In [ ]:
# df_bpi=pd.read_csv(r"C:\Users\vikram.vadhirajan\OneDrive - Trico\Documents - Product Support India Romania\Catalog\Score card\EPR\20250416\Sales\Product to Customer BPI Centric ASC IBI.csv")
# df_bpi

,BusinessUnit,OrganizationDesc,CustomerCode,CustomerDesc,CustomerAddress1,CustomerState,CustomerZip,ProductCode,ProductDesc,QTY SHIPPED
0,BPI,MIDWEST DC,1986002,GMSPO WD SHIP TO,4400 Prime Parkway,IL,60050,96-18J4091,BRAKE HOSE,36
1,BPI,MIDWEST DC,2535ST,NAPA WASHINGTON DC,QUAKER CITY MOTOR PARTS,DE,19709,62-683052,BRAKE HOSE,99
2,BPI,MIDWEST DC,Z7097,UNITED AUTO SUPPLY,1200 STATE FAIR BLVD,NY,13209,MC390938,MASTER CYLINDER,2
3,BPI,MIDWEST DC,2120ST,NAPA DENVER,GENUINE PARTS COMPANY,GA,30091-2227,62-48880680SP,BRAKE ROTOR,100
4,BPI,MIDWEST DC,2295ST,NAPA MT VERNON,GENUINE PARTS COMPANY,GA,30091-0349,62-86934CR,BRAKE ROTOR,30
...,...,...,...,...,...,...,...,...,...,...
1221439,BPI,MIDWEST DC,21339,"PARTS WHOLESALERS, INC.",125 S. WALNUT ST,WA,99204,EHT1391AH,PAD SET IM,1
1221440,BPI,MIDWEST DC,40002055,NAPA CARROLLTON DC,"1233 Lincoln Ave, N.W.",OH,44615,62-682897,BRAKE HOSE,1
1221441,Centric,CENTRIC (ORACLE),C2647,TURN 14 DISTRIBUTION,100 TOURNAMENT DRIVE,PA,19044,105.09140,PAD SET IM,1
1221442,BPI,MIDWEST DC,32068011,MODERN SALES CO-OP,293057 JAMES JONES WAY,AB,T4A 0X1,H17344,DRUM BRAKE HDWE,1


In [ ]:
# df_Framauto=pd.read_excel(r"C:\Users\vikram.vadhirajan\OneDrive - Trico\Documents - Product Support India Romania\Catalog\Score card\EPR\20250416\Sales\FramAutolite.xlsx",sheet_name="Sheet3")
# df_Framauto

""


In [ ]:
df_champ=pd.read_excel(r"C:\Users\vikram.vadhirajan\OneDrive - Trico\Documents - Product Support India Romania\Catalog\Score card\EPR\20250416\Sales\Champ.xlsx")
df_champ

,Ship To,Ship To Name,Ordered As,Part#,Part\nDescription,Qty Ordered,State
0,427769,THE PARTS CONNECTION,/46367202,0002470904632,PARTS MASTER OIL FILTER GB,12,VA
1,427769,THE PARTS CONNECTION,/46361311,0002499724630,PARTS MASTER OIL FILTER,12,VA
2,427769,THE PARTS CONNECTION,/46367033,0003267044630,PARTS MASTER OIL FILTER,12,VA
3,427769,THE PARTS CONNECTION,/46367211,0003275414630,PARTS MASTER OIL FILTER,12,VA
4,427769,THE PARTS CONNECTION,/46361212,0003444304630,PARTS MASTER OIL FILTER,12,VA
...,...,...,...,...,...,...,...
528553,479167,MR LUBE #308,/590PH51A (N),0002524515900,LUBER-FINER OIL FILTER,12,BC
528554,479167,MR LUBE #308,/590P1051,0003472306890,LUBERFINER OIL FILTER,1,BC
528555,479167,MR LUBE #308,/590P8170,0003494986890,LUBERFINER OIL FILTER WB,1,BC
528556,479167,MR LUBE #308,/590P978,0003509315900,LUBERFINER OIL FILTER,12,BC
